In [ ]:
import polars as pl
from procompa import get_project_root, clean_identifiers

PRJ_ROOT = get_project_root()
data_dir = PRJ_ROOT / "data"

In [ ]:
Complex_portal_df = pl.read_csv("/cluster/project/beltrao/kdammer/master_thesis/data/Complex_Portal/Saccharomyces_cerevisiae_ComplexTab.tsv", separator="\t")
#Complex_portal_df.select(pl.col("#Complex ac"), pl.col("Cross references"), pl.col("Subunits (UniProt IDs)")).head(5)

In [4]:
# === Enrich CombFold results CSV with PDB availability + exact-match info ===
import pandas as pd, re

# --- Paths (edit these to match your cluster layout) ---
combfold_csv = data_dir/"Pipeline/third_setup/pdb_present_for_stoi_gr_two_third_setup_pipeline_complexes_combfold_results.csv"
tsv_path     = data_dir/ "Complex_Portal/Saccharomyces_cerevisiae_ComplexTab.tsv"
exact_csv    = data_dir/"complete_complex_pdb_mapping/uniprot_pdb/complex_pdb_exact_match.csv"
out_csv      = data_dir/"complete_complex_pdb_mapping/pdb_present_for_stoi_gr_two_third_setup_pipeline_complexes_combfold_results_enriched.csv"

# --- 1. Load CombFold results ---
cf = pd.read_csv(combfold_csv)
print(f"CombFold CSV: {len(cf)} rows, columns: {list(cf.columns)}")

# Find the complex-ID column (try common names)
cpx_col = None
for candidate in ["#Complex ac", "complex_accession", "Complex ac", "CPX"]:
    if candidate in cf.columns:
        cpx_col = candidate
        break
if cpx_col is None:
    raise SystemExit(f"Could not find complex-ID column. Available: {list(cf.columns)}")
print(f"Using complex-ID column: '{cpx_col}'")

# --- 2. Parse Complex Portal TSV for identity/subset/experimental_evidence tags ---
tsv = pd.read_csv(tsv_path, sep="	")
WWPDB_TAGGED = re.compile(r"^wwpdb:([A-Za-z0-9]{4})\((identity|subset)\)$")
WWPDB_BARE   = re.compile(r"\bwwpdb:([A-Za-z0-9]{4})\b")

cpx_pdb_info = {}  # cpx -> {"tags": set, "pdb_ids": set}
for _, row in tsv.iterrows():
    cpx = str(row["#Complex ac"]).strip()
    if not cpx:
        continue
    tags = set()
    pdbs = set()
    xref = row.get("Cross references", "")
    if isinstance(xref, str):
        for tok in xref.split("|"):
            m = WWPDB_TAGGED.match(tok.strip())
            if m:
                tags.add(m.group(2))
                pdbs.add(m.group(1).upper())
    expev = row.get("Experimental evidence", "")
    if isinstance(expev, str):
        tagged = pdbs.copy()
        for match in WWPDB_BARE.finditer(expev):
            pid = match.group(1).upper()
            if pid not in tagged:
                tags.add("experimental_evidence")
                pdbs.add(pid)
    cpx_pdb_info[cpx] = {"tags": tags, "pdb_ids": pdbs}

# --- 3. Load exact-match CSV ---
exact = pd.read_csv(exact_csv)
exact_info = {}  # cpx -> {"has_exact": bool, "n_exact": int, "pdbs": str}
for _, row in exact.iterrows():
    cpx = row["complex_accession"]
    if row["match_class"] == "exact_match":
        n = row["n_exact_match_pdbs"]
        pdbs = row["all_exact_pdbs"] if pd.notna(row["all_exact_pdbs"]) else ""
        exact_info[cpx] = {"has_exact": True, "n_exact": int(n), "pdbs": pdbs}
    else:
        exact_info[cpx] = {"has_exact": False, "n_exact": 0, "pdbs": ""}

# --- 4. Build enrichment columns ---
has_pdb_col, tags_col, pdb_ids_col = [], [], []
has_exact_col, n_exact_col, exact_pdbs_col = [], [], []

for cpx in cf[cpx_col]:
    cpx = str(cpx).strip()
    info = cpx_pdb_info.get(cpx, {"tags": set(), "pdb_ids": set()})
    has_pdb_col.append("yes" if info["pdb_ids"] else "no")
    tags_col.append(";".join(sorted(info["tags"])) if info["tags"] else "none")
    pdb_ids_col.append(";".join(sorted(info["pdb_ids"])))

    ex = exact_info.get(cpx, {"has_exact": False, "n_exact": 0, "pdbs": ""})
    has_exact_col.append("yes" if ex["has_exact"] else "no")
    n_exact_col.append(ex["n_exact"])
    exact_pdbs_col.append(ex["pdbs"])

cf["has_complex_portal_pdb"]   = has_pdb_col
cf["complex_portal_tags"]      = tags_col
cf["complex_portal_pdb_ids"]   = pdb_ids_col
cf["has_exact_match"]          = has_exact_col
cf["n_exact_match_pdbs"]       = n_exact_col
cf["exact_match_pdb_ids"]      = exact_pdbs_col

# --- 5. Save + summary ---
cf.to_csv(out_csv, index=False)
print(f"Enriched CSV written: {out_csv}")
print(f"=== Summary ===")
print(f"  Total complexes in CombFold CSV: {len(cf)}")
print(f"  Has Complex Portal PDB (any tag): {(cf['has_complex_portal_pdb']=='yes').sum()}")
print(f"    of which identity:        {cf['complex_portal_tags'].str.contains('identity').sum()}")
print(f"    of which subset:          {cf['complex_portal_tags'].str.contains('subset').sum()}")
print(f"    of which experimental_evidence: {cf['complex_portal_tags'].str.contains('experimental_evidence').sum()}")
print(f"  Has SIFTS exact match:      {(cf['has_exact_match']=='yes').sum()}")
print(f"  Has neither:                {((cf['has_complex_portal_pdb']=='no') & (cf['has_exact_match']=='no')).sum()}")

CombFold CSV: 116 rows, columns: ['true_complex', 'predicted_complex', 'size_true', 'size_pred', 'match_count', 'jaccard_similarity', 'split', 'exact_size_match', '#Complex ac', 'confidence_score', 'ComplexConfidence', 'Identifiers (and stoichiometry) of molecules in complex', 'stoichiometry_known', 'solely_proteins', 'comb_fold_submission', 'proteins_with_homodimer_pdb', 'pdb_for_true_stoi', 'combfold_successfully', 'n_assembled_outputs', 'confidence_scores']
Using complex-ID column: '#Complex ac'
Enriched CSV written: /cluster/project/beltrao/kdammer/master_thesis/data/complete_complex_pdb_mapping/pdb_present_for_stoi_gr_two_third_setup_pipeline_complexes_combfold_results_enriched.csv
=== Summary ===
  Total complexes in CombFold CSV: 116
  Has Complex Portal PDB (any tag): 47
    of which identity:        33
    of which subset:          19
    of which experimental_evidence: 4
  Has SIFTS exact match:      46
  Has neither:                61
